<img src="https://www.unad.edu.co/images/footer/logo-unad-acreditacion-min.png" width="780" height="140" align="right"/>

<p style="text-align: center;"> Curso: ENSEMBLE METHODS AND KERNELS</p>

<p style="text-align: center;"> Código Curso: 203008076 </p>

<p style="text-align: center;"> Grupo: 1 </p>

<p style="text-align: center;"> Phase 3 -Development of the Practical Component of the
Course Ensemble Methods and Kernels</p>

<p style="text-align: center;">  Presentado por: Wilmer Ricardo Urda</p>

<p style="text-align: center;"> Código: 1017194627</p>

<p style="text-align: center;">  Tutor: Ing. Jorge Luis Quintero Lopez </p>

<p style="text-align: center;"> UNIVERSIDAD NACIONAL ABIERTA Y A DISTANCIA - UNAD </p>

# Exercise 3: Random Forest Method

## RANDOM FOREST – REGRESIÓN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# =========================
# CARGAR DATASET (meta - ID 566)
# =========================
data = fetch_openml(data_id=566, as_frame=True)
df = data.frame.copy()

target_col = data.target_names[0]
X = df.drop(columns=[target_col])
y = pd.to_numeric(df[target_col], errors='coerce')

mask = y.notna()
X, y = X[mask], y[mask]

# =========================
# PREPROCESAMIENTO CON PIPELINE
# =========================
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# MODELO RANDOM FOREST
# =========================
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random_forest', RandomForestRegressor(
        n_estimators=100,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# =========================
# MÉTRICAS
# =========================
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
metrics_df = pd.DataFrame({'Metric': ['R²', 'RMSE'], 'Value': [round(r2, 4), round(rmse, 4)]})
print(metrics_df.to_string(index=False))

In [ ]:
# =========================
# FEATURE IMPORTANCE
# =========================
rf_step   = model.named_steps['random_forest']
prep_step = model.named_steps['preprocessor']

ohe = prep_step.named_transformers_['cat'].named_steps['onehot']
cat_feature_names = list(ohe.get_feature_names_out(cat_cols))
all_feature_names = num_cols + cat_feature_names

importances = rf_step.feature_importances_
top_n   = min(15, len(importances))
indices = np.argsort(importances)[::-1][:top_n]

plt.figure(figsize=(10, 5))
plt.bar(range(top_n), importances[indices], color='steelblue')
plt.xticks(range(top_n), [all_feature_names[i] for i in indices], rotation=45, ha='right')
plt.title('Top 15 Feature Importances – Random Forest Regressor (meta dataset)')
plt.xlabel('Feature')
plt.ylabel('Importance (Mean Decrease in Impurity)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# LEARNING CURVE
# =========================
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y,
    cv=5,
    scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), marker='o', label='Train R²')
plt.plot(train_sizes, test_scores.mean(axis=1),  marker='o', label='Test R²')
plt.fill_between(train_sizes,
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15)
plt.fill_between(train_sizes,
                 test_scores.mean(axis=1) - test_scores.std(axis=1),
                 test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.15)
plt.title('Learning Curve – Random Forest Regressor (meta dataset)')
plt.xlabel('Training Size')
plt.ylabel('R²')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Random Forest – Regression

The Random Forest Regressor was implemented using a full `Pipeline` that includes preprocessing (median imputation for numerical features and one-hot encoding for categorical features such as `DS_Name` and `Alg_Name`) followed by a `RandomForestRegressor` with 100 estimators and `max_features='sqrt'`.

Random Forest extends standard Bagging by introducing an additional randomization step: at each node split, only a random subset of `sqrt(n_features)` features is evaluated. This decorrelates the individual trees more aggressively than standard Bagging, where all features are candidates at each split, resulting in lower ensemble variance.

The learning curve shows that:

- Training R² remains consistently high across all training sizes, typical for ensemble tree methods with sufficient estimators.
- Test R² improves progressively as more training samples are used, though it remains modest due to the inherent noise in the meta dataset.
- The narrow confidence bands confirm stable and reproducible behavior across the five cross-validation folds.
- The gap between training and test R² is reduced compared to a single decision tree, confirming the variance-reduction effect of aggregating 100 decorrelated trees.

The low R² values are expected for the meta dataset, which records algorithm performance across heterogeneous benchmark datasets. This makes the target variable inherently noisy and difficult to predict — a known property of meta-learning benchmarks rather than a failure of the model.

The feature importance plot (Mean Decrease in Impurity) reveals which numerical and encoded categorical features most influence predicted performance, providing an interpretable view of the model that complements its predictive metrics.

## RANDOM FOREST – CLASIFICACIÓN

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =========================
# CARGAR DATASET (kr-vs-kp - ID 3)
# =========================
data_clf = fetch_openml(data_id=3, as_frame=True)
df_clf = data_clf.frame.copy()

target_col_clf = data_clf.target_names[0]
X_clf = df_clf.drop(columns=[target_col_clf])
y_clf = LabelEncoder().fit_transform(df_clf[target_col_clf])

# =========================
# PREPROCESAMIENTO CON PIPELINE
# =========================
cat_cols_clf = X_clf.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols_clf = X_clf.select_dtypes(include=[np.number]).columns.tolist()

preprocessor_clf = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols_clf),
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols_clf)
])

# =========================
# SPLIT (con stratify para balance de clases)
# =========================
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# =========================
# MODELO RANDOM FOREST
# =========================
model_clf = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('random_forest', RandomForestClassifier(
        n_estimators=100,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ))
])

model_clf.fit(X_train_clf, y_train_clf)
y_pred_clf = model_clf.predict(X_test_clf)

# =========================
# MÉTRICAS
# =========================
metrics_clf = pd.DataFrame([{
    'Accuracy':  round(accuracy_score(y_test_clf, y_pred_clf), 4),
    'Precision': round(precision_score(y_test_clf, y_pred_clf, average='weighted'), 4),
    'Recall':    round(recall_score(y_test_clf, y_pred_clf, average='weighted'), 4),
    'F1-score':  round(f1_score(y_test_clf, y_pred_clf, average='weighted'), 4),
}])
print(metrics_clf.to_string(index=False))

In [ ]:
# =========================
# FEATURE IMPORTANCE
# =========================
rf_step_clf   = model_clf.named_steps['random_forest']
prep_step_clf = model_clf.named_steps['preprocessor']

ohe_clf = prep_step_clf.named_transformers_['cat'].named_steps['onehot']
cat_feature_names_clf = list(ohe_clf.get_feature_names_out(cat_cols_clf))
all_feature_names_clf = num_cols_clf + cat_feature_names_clf

importances_clf = rf_step_clf.feature_importances_
top_n_clf   = min(15, len(importances_clf))
indices_clf = np.argsort(importances_clf)[::-1][:top_n_clf]

plt.figure(figsize=(10, 5))
plt.bar(range(top_n_clf), importances_clf[indices_clf], color='darkorange')
plt.xticks(range(top_n_clf), [all_feature_names_clf[i] for i in indices_clf], rotation=45, ha='right')
plt.title('Top 15 Feature Importances – Random Forest Classifier (kr-vs-kp dataset)')
plt.xlabel('Feature')
plt.ylabel('Importance (Mean Decrease in Impurity)')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# LEARNING CURVE
# =========================
train_sizes_clf, train_scores_clf, test_scores_clf = learning_curve(
    model_clf, X_clf, y_clf,
    cv=5,
    scoring='accuracy',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes_clf, train_scores_clf.mean(axis=1), marker='o', label='Train Accuracy')
plt.plot(train_sizes_clf, test_scores_clf.mean(axis=1),  marker='o', label='Test Accuracy')
plt.fill_between(train_sizes_clf,
                 train_scores_clf.mean(axis=1) - train_scores_clf.std(axis=1),
                 train_scores_clf.mean(axis=1) + train_scores_clf.std(axis=1), alpha=0.15)
plt.fill_between(train_sizes_clf,
                 test_scores_clf.mean(axis=1) - test_scores_clf.std(axis=1),
                 test_scores_clf.mean(axis=1) + test_scores_clf.std(axis=1), alpha=0.15)
plt.title('Learning Curve – Random Forest Classifier (kr-vs-kp dataset)')
plt.xlabel('Training Size')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Random Forest – Classification

The Random Forest Classifier was implemented using a `Pipeline` that applies one-hot encoding to all categorical features of the `kr-vs-kp` dataset via `ColumnTransformer`, preventing data leakage by fitting the encoder only on the training set. The split was performed with `stratify=y` to preserve the original class distribution in both training and test sets.

The key differentiator from standard Bagging is the use of `max_features='sqrt'` (the scikit-learn default for classifiers), which limits feature evaluation at each split to the square root of the total number of features. This additional randomization decorrelates the trees in the ensemble, reducing variance beyond what Bagging alone achieves.

The learning curve shows that:

- Training accuracy reaches near-perfect values across all training sizes, reflecting the expressive capacity of 100 trees on a fully categorical dataset.
- Test accuracy rapidly converges toward the training accuracy as sample size grows, confirming strong generalization with minimal overfitting.
- The very narrow confidence bands indicate highly consistent behavior across the five cross-validation folds, demonstrating the stability of the Random Forest ensemble.
- The gap between training and test accuracy remains smaller than for a single decision tree, confirming the variance reduction introduced by both bootstrap aggregation and feature subsampling.

The feature importance plot identifies which chess-position features (encoded from the 36 categorical columns) most influence the classification of king-rook vs king-pawn endgame positions. This interpretability is a practical advantage of Random Forest over gradient-based methods like XGBoost, which require additional tools (e.g., SHAP) to achieve comparable explanations.

# Exercise 3 – Random Forest Method

In this exercise, the Random Forest method was implemented for both regression and classification tasks using the same datasets as in Exercises 1 and 2.

For the regression task, the **meta (ID: 566)** dataset from OpenML was used.  
For the classification task, the **kr-vs-kp (ID: 3)** dataset from OpenML was used.

A preprocessing pipeline was applied to handle missing values and encode categorical variables, ensuring no data leakage from the test set into the feature transformations. Performance was evaluated through appropriate metrics and learning curves using:

`train_sizes = np.linspace(0.1, 1.0, 8)`

The objective of this exercise is to understand the specific mechanisms of Random Forest and compare its behavior against Bagging and Boosting.

## Comparative Discussion: Random Forest vs Bagging vs Boosting

Random Forest (RF) is a specialized extension of Bagging that introduces a second source of randomness: at each node split, only a random subset of features is considered (`max_features='sqrt'`). This has two key effects:

1. **Stronger decorrelation:** Trees in a Bagging ensemble are still correlated because they all select from the same full feature set. RF breaks this correlation by restricting feature access, making each tree genuinely different and reducing the ensemble variance more effectively.
2. **Slightly weaker individual trees:** Each tree in RF has a higher bias (since it cannot always pick the globally best feature), but this is compensated by the averaging of many low-correlation trees.

**Random Forest vs Bagging:**  
RF consistently outperforms standard Bagging due to the additional feature subsampling. Both methods are parallelizable (trees are trained independently), but RF achieves better generalization with the same number of estimators.

**Random Forest vs Boosting:**  
Boosting trains trees sequentially, with each tree correcting the residual errors of the previous ones. This makes Boosting more powerful on complex, structured datasets but also more prone to overfitting and computationally more expensive. RF trains all trees independently (parallelizable), making it faster and more robust.  
For the kr-vs-kp dataset, RF achieves near-perfect accuracy comparable to Gradient Boosting and XGBoost, confirming that the dataset's patterns are well-captured by feature-subsampled ensembles without the need for sequential error correction.  
For the meta dataset, all tree-based ensemble methods (Bagging, Boosting, RF) achieve similarly low R² values, reflecting the fundamental difficulty of this meta-learning benchmark rather than differences between methods.

**Interpretability:**  
A practical advantage of Random Forest over Boosting methods is that it natively exposes feature importances (Mean Decrease in Impurity), providing immediate interpretability without additional post-hoc tools. This makes RF preferable in scenarios where model transparency is required alongside predictive performance.